# Part B: lever 2 (soft span targets) on ModernBERT-base

Two training runs that differ only in the targets: `mb_hard` uses 0/1 token labels (as LettuceDetect does),
`mb_soft` uses the 2021-style soft span targets. Same model, data, order, seed and settings. 10% of the
training sources are held out as a dev set to pick the epoch and to fit levers 1 and 3.

About 30-45 minutes per run on a T4. Each run saves its predictions to Drive; if Colab disconnects,
run the cells again and finished runs are skipped.

Choose **Runtime > Change runtime type > T4 GPU** first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
PROJECT = '/content/drive/MyDrive/hallu_poc'   # the folder with poc.py
WORK = f'{PROJECT}/work'                        # results are saved here
DATA = f'{PROJECT}/RAGTruth'
if not os.path.exists(f'{DATA}/response.jsonl') and not os.path.exists(f'{DATA}/dataset/response.jsonl'):
    DATA = '/content/RAGTruth'                   # not on Drive: get it from GitHub
    if not os.path.exists(DATA):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth', DATA], check=True)
print('data:', DATA)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ModernBERT needs transformers 4.48 or newer
import transformers
from packaging import version
if version.parse(transformers.__version__) < version.parse('4.48'):
    !pip -q install -U 'transformers>=4.48'
    print('Updated transformers: Runtime > Restart session, then run the cells again.')
else:
    print('transformers', transformers.__version__)

## 1. Train both runs

In [ ]:
EPOCHS = 3
for targets in ('hard', 'soft'):
    name = f'mb_{targets}'
    if os.path.exists(f'{WORK}/preds/{name}_test.pkl'):
        print(name, 'already done')
        continue
    !cd {PROJECT} && python -u poc.py train --data-dir {DATA} --work-dir {WORK} \
        --base answerdotai/ModernBERT-base --targets {targets} --name {name} --epochs {EPOCHS}

## 2. Levers 1 and 3, fitted on the dev set

In [ ]:
for name in ('mb_hard', 'mb_soft'):
    !cd {PROJECT} && python -u poc.py levers --data-dir {DATA} --work-dir {WORK} --name {name} --fit dev

## 3. All results, and soft minus hard

In [ ]:
!cd {PROJECT} && python -u poc.py report --data-dir {DATA} --work-dir {WORK} --compare mb_hard mb_soft

**Reading it.** The last lines give span F1 of `mb_soft` minus `mb_hard` on the same test answers, with a
95% bootstrap interval. If the interval stays above 0, soft targets helped. One seed is a first signal,
not proof: a second seed per run would tell how much of the difference is noise.